# Phase 5a - deep pilot on GPU (inner folds of repeat 0, outer fold 0 only)

GPU verification, model code (`src/soh_deep.py`), parallel multi-model trainer, tests, speed benchmark, inner-fold pilot, and a compute plan for Phase 5b.
No outer-test cell, batch-3 cell or sensitivity cell is scored or loaded into any array. No gate file is written. fp32 only, TF32 off, no pretrained weights.

## 1. Setup, GPU verification, guards

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # must precede the torch import
import sys, json, time, math, gc, re, hashlib, inspect, warnings, traceback, contextlib, itertools, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import torch
T_START = time.time()

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
PROC = ROOT / "data/processed"; SPL = PROC / "splits"
P3 = ROOT / "outputs/phase3"; OUT4 = ROOT / "outputs/phase4"          # OUT4 is READ-ONLY
OUT = ROOT / "outputs/phase5a"; FIG = OUT / "figures"; CACHE = OUT / "cache"; SRC = ROOT / "src"
for d in (OUT, FIG, CACHE):
    d.mkdir(parents=True, exist_ok=True)
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

SEED = 42; NOMINAL_AH = 1.1; TRAIN_STRIDE = 5; BATCH = 256; CLIP = 1.0
HUBER_DELTA = 1.0; WD = 1e-4; NOISE_SIGMA = 0.02; PILOT_WINDOWS = ["W300", "W240"]
WINDOWS = ["W310", "W305", "W300", "W280", "W240", "WFULL"]
WINDOW_END_V = {"W310": 3.10, "W305": 3.05, "W300": 3.00, "W280": 2.80, "W240": 2.40, "WFULL": 2.00}
GATE_TARGETS = {"W310": 0.9238, "W305": 0.8046, "W300": 0.7605, "W280": 0.6784, "W240": 0.5569, "WFULL": 0.5319}
EXPECTED_GATE_HASH = "40df2fb82096e994f8bf7b8084433bb5ca148d3fa92c780461f68c0c4247d901"
CODE_VERSION = "p5a-v1"      # part of every cache key; bump to invalidate caches

CHECKS, ERRORS = [], []
def check(name, passed, detail="", level="FAIL"):
    status = "PASS" if passed else level
    CHECKS.append(dict(name=name, status=status, detail=str(detail)))
    print(f"[{status}] {name}: {detail}")
    return bool(passed)

@contextlib.contextmanager
def guard(name):
    # record the error and continue; never swallow silently
    try:
        yield
    except Exception as e:
        ERRORS.append(dict(where=name, error=repr(e), trace=traceback.format_exc()[-1800:]))
        CHECKS.append(dict(name=name, status="FAIL", detail=repr(e)))
        print(f"[FAIL] {name}: {e!r}")

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

def col(df, cands, what):
    low = {c.lower(): c for c in df.columns}
    for c in cands:
        if c.lower() in low:
            return low[c.lower()]
    raise KeyError(f"{what}: none of {cands} in columns {list(df.columns)}")

def flex_call(fn, ctx):
    # call fn by mapping its parameter names onto ctx; raise if a required parameter is unmapped
    sig = inspect.signature(fn); kw = {}; missing = []
    for n, p in sig.parameters.items():
        if n in ctx:
            kw[n] = ctx[n]
        elif p.default is inspect._empty and p.kind in (p.POSITIONAL_OR_KEYWORD, p.KEYWORD_ONLY):
            missing.append(n)
    if missing:
        raise TypeError(f"{fn.__name__}{sig}: unmapped required parameters {missing}; ctx keys {sorted(ctx)}")
    return fn(**kw)

def jd(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (set, tuple)): return list(o)
    return str(o)

print("ROOT:", ROOT)

ROOT: c:\workstation\3RD YEAR\Battery-Health-Estimation


In [2]:
# GPU verification, determinism, fp32 only
import torch.nn as nn
CUDA_OK = torch.cuda.is_available()
check("CUDA available", CUDA_OK, "required")
if not CUDA_OK:
    raise RuntimeError("CUDA is unavailable; Phase 5a needs the GPU. Fix the environment and rerun.")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
dev = torch.device("cuda")
pr = torch.cuda.get_device_properties(0)
free_b, total_b = torch.cuda.mem_get_info()
GPU_FACTS = dict(torch=torch.__version__, cuda=torch.version.cuda, cudnn=str(torch.backends.cudnn.version()),
                 device=pr.name, capability=f"{pr.major}.{pr.minor}", vram_total_gb=round(total_b / 1e9, 2),
                 vram_free_gb=round(free_b / 1e9, 2), tf32_matmul=torch.backends.cuda.matmul.allow_tf32,
                 tf32_cudnn=torch.backends.cudnn.allow_tf32)
for k, v in GPU_FACTS.items():
    print(f"{k:14s} {v}")
check("TF32 off", not GPU_FACTS["tf32_matmul"] and not GPU_FACTS["tf32_cudnn"])

# CPU vs GPU parity for a small conv + linear
torch.manual_seed(0)
_conv, _lin = nn.Conv1d(1, 8, 9, 2), nn.Linear(8, 1)
_x = torch.randn(64, 1, 200)
_cpu = _lin(_conv(_x).mean(-1))
_gpu = _lin.to(dev)(_conv.to(dev)(_x.to(dev)).mean(-1)).cpu()
_d = (_cpu - _gpu).abs().max().item()
check("CPU-vs-GPU parity (conv+linear)", _d < 1e-5, f"max abs diff {_d:.2e}")

try:
    import psutil
    _ram = psutil.virtual_memory().available / 1e9
    check("available RAM >= 2 GB", _ram >= 2.0, f"{_ram:.1f} GB", level="WARN")
except Exception as e:
    check("psutil RAM check", False, repr(e), level="WARN")

# frozen gate integrity (re-checked at the very end)
GATE_FILE = OUT4 / "week2_gate.json"
_m = re.search(r"[0-9a-f]{64}", (OUT4 / "gate_hash.txt").read_text())
GATE_HASH_FILE = _m.group(0) if _m else ""
def gate_ok():
    return sha256_file(GATE_FILE) == GATE_HASH_FILE
check("week2_gate.json sha256 == gate_hash.txt (start)", gate_ok(), GATE_HASH_FILE[:16] + "...")
check("gate_hash.txt equals expected frozen hash", GATE_HASH_FILE == EXPECTED_GATE_HASH, "", level="WARN")
GATE_JSON = json.loads(GATE_FILE.read_text())
def _leaves(o):
    if isinstance(o, dict):
        for v in o.values(): yield from _leaves(v)
    elif isinstance(o, (list, tuple)):
        for v in o: yield from _leaves(v)
    elif isinstance(o, (int, float)) and not isinstance(o, bool):
        yield float(o)
_nums = list(_leaves(GATE_JSON))
for w, t in GATE_TARGETS.items():
    check(f"gate target {w} = {t} present in week2_gate.json", any(abs(v - t) < 5e-4 for v in _nums), "", level="WARN")

[PASS] CUDA available: required
torch          2.14.1+cu130
cuda           13.0
cudnn          92400
device         NVIDIA GeForce RTX 3050 6GB Laptop GPU
capability     8.6
vram_total_gb  6.44
vram_free_gb   5.41
tf32_matmul    False
tf32_cudnn     False
[PASS] TF32 off: 
[PASS] CPU-vs-GPU parity (conv+linear): max abs diff 4.47e-08
[WARN] available RAM >= 2 GB: 0.6 GB
[PASS] week2_gate.json sha256 == gate_hash.txt (start): 40df2fb82096e994...
[PASS] gate_hash.txt equals expected frozen hash: 
[PASS] gate target W310 = 0.9238 present in week2_gate.json: 
[PASS] gate target W305 = 0.8046 present in week2_gate.json: 
[PASS] gate target W300 = 0.7605 present in week2_gate.json: 
[PASS] gate target W280 = 0.6784 present in week2_gate.json: 
[PASS] gate target W240 = 0.5569 present in week2_gate.json: 
[PASS] gate target WFULL = 0.5319 present in week2_gate.json: 


In [3]:
# soh_common helpers: print signatures, state which are used
import soh_common as sc
HELPERS = ["load_tables", "grid_index", "window_slice", "assert_allowed", "build_delta_q", "window_valid", "shuffle_labels_within_cell"]
for n in HELPERS:
    if hasattr(sc, n):
        print(f"{n}{inspect.signature(getattr(sc, n))}")
    else:
        print(f"{n}: not present in soh_common")
print("Used here: window_slice (window widths), build_delta_q (3-cell recompute check), assert_allowed (cell guard, if mappable), grid_index (fallback widths).")
print("Not used: load_tables (tables are read directly, restricted to training cells), window_valid (finite-row filter instead), shuffle_labels_within_cell (no label-shuffle control in 5a).")
SC_FILE_HASH = sha256_file(Path(sc.__file__))

load_tables(root=None, mmap=True)
grid_index(voltage)
window_slice(open_v, close_v)
assert_allowed(columns, allowlist=None, roles=('primary', 'ablation'))
build_delta_q(qdlin_row, qref, i_open, i_close, n_points=None)
window_valid(qdlin_row, i_open, i_close, max_step_ah=0.05)
shuffle_labels_within_cell(df, seed, label_col='soh_nom', cell_col='cell_id')
Used here: window_slice (window widths), build_delta_q (3-cell recompute check), assert_allowed (cell guard, if mappable), grid_index (fallback widths).
Not used: load_tables (tables are read directly, restricted to training cells), window_valid (finite-row filter instead), shuffle_labels_within_cell (no label-shuffle control in 5a).


In [4]:
# Write src/soh_deep.py once (never overwrite), then import it
MODULE_SRC = r'''"""soh_deep: small 1D-CNN (M1) and MLP (M0) for capacity-free SOH nowcasting.

Phase 5a module. Contents:
  * M1, M0            ordinary single models (no pretrained weights anywhere).
  * BatchedM1/M0      G independent members trained at once (grouped conv, per-member BN,
                      per-member linear layers via einsum). Initialised by stacking G ordinary
                      single models, so member g starts identical to the single model of seed g.
  * Ensemble          wrapper with two packings: "batched" or "sequential" (list of singles).
  * fit               parallel multi-model trainer (own rows / weights / seed / head per member).
  * make_train_rows   strided, cell-balanced training-row recipe.

fp32 only. No weights are ever loaded from or saved to disk.
"""
from __future__ import annotations

import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

NOMINAL_AH = 1.1


# --------------------------------------------------------------------------- helpers
def _rnd(x: float, lo: int = 2) -> int:
    return max(lo, int(round(x)))


def m1_widths(c: float = 1.0):
    """Channel widths (conv1, conv2, conv3, head) for width multiplier c."""
    return _rnd(8 * c), _rnd(16 * c), _rnd(32 * c), _rnd(16 * c)


def sub_idx(L: int, n: int = 32) -> np.ndarray:
    """n points spread over a window of length L (np.linspace, as the Phase 4 M0/B4 subsample)."""
    return np.round(np.linspace(0, L - 1, n)).astype(np.int64)


def n_params(model: nn.Module) -> int:
    return int(sum(p.numel() for p in model.parameters()))


# --------------------------------------------------------------------------- single models
class M1(nn.Module):
    """Conv1d(1->a,k9,s2)-BN-ReLU-Conv1d(a->b,k5,s2)-BN-ReLU-Conv1d(b->d,k3,s2)-BN-ReLU-GAP
    -> concat q0 -> Linear(d+1->h)-ReLU-Linear(h->1). 2,977 parameters at c=1."""

    def __init__(self, c: float = 1.0):
        super().__init__()
        a, b, d, h = m1_widths(c)
        self.c1, self.b1 = nn.Conv1d(1, a, 9, 2), nn.BatchNorm1d(a)
        self.c2, self.b2 = nn.Conv1d(a, b, 5, 2), nn.BatchNorm1d(b)
        self.c3, self.b3 = nn.Conv1d(b, d, 3, 2), nn.BatchNorm1d(d)
        self.f1, self.f2 = nn.Linear(d + 1, h), nn.Linear(h, 1)

    def forward(self, x, q):
        """x: (B, L) scaled delta-Q; q: (B,) standardized q0 -> (B,) network output z."""
        h = x.unsqueeze(1)
        h = F.relu(self.b1(self.c1(h)))
        h = F.relu(self.b2(self.c2(h)))
        h = F.relu(self.b3(self.c3(h)))
        h = torch.cat([h.mean(-1), q.unsqueeze(1)], 1)
        return self.f2(F.relu(self.f1(h))).squeeze(1)


class M0(nn.Module):
    """MLP on a 32-point subsample of delta-Q + q0; hidden (44, 32); 2,969 parameters."""

    def __init__(self, L: int, n_pts: int = 32, hidden=(44, 32)):
        super().__init__()
        self.register_buffer("idx", torch.as_tensor(sub_idx(L, n_pts)))
        self.l1 = nn.Linear(n_pts + 1, hidden[0])
        self.l2 = nn.Linear(hidden[0], hidden[1])
        self.l3 = nn.Linear(hidden[1], 1)

    def forward(self, x, q):
        h = torch.cat([x[:, self.idx], q.unsqueeze(1)], 1)
        return self.l3(F.relu(self.l2(F.relu(self.l1(h))))).squeeze(1)


# --------------------------------------------------------------------------- batched models
def _cat_param(singles, getter):
    return nn.Parameter(torch.cat([getter(s).detach().clone() for s in singles], 0))


def _lin_w(singles, getter):  # (G, in, out)
    return nn.Parameter(torch.stack([getter(s).detach().t().contiguous() for s in singles], 0))


def _lin_b(singles, getter):  # (G, out)
    return nn.Parameter(torch.stack([getter(s).detach().clone() for s in singles], 0))


def _lin(h, w, b):
    return torch.einsum("gbi,gio->gbo", h, w) + b[:, None, :]


class BatchedM1(nn.Module):
    """G independent M1 members: grouped Conv1d (groups=G), BN over G*C channels (so per-member
    statistics), per-member linear layers as einsum with weights (G, in, out)."""

    def __init__(self, singles):
        super().__init__()
        self.G = len(singles)
        s0 = singles[0]
        self.a, self.b, self.d = s0.c1.out_channels, s0.c2.out_channels, s0.c3.out_channels
        for n in ("c1", "c2", "c3"):
            setattr(self, n + "_weight", _cat_param(singles, lambda s, n=n: getattr(s, n).weight))
            setattr(self, n + "_bias", _cat_param(singles, lambda s, n=n: getattr(s, n).bias))
        for n in ("b1", "b2", "b3"):
            setattr(self, n + "_weight", _cat_param(singles, lambda s, n=n: getattr(s, n).weight))
            setattr(self, n + "_bias", _cat_param(singles, lambda s, n=n: getattr(s, n).bias))
            self.register_buffer(n + "_rm", torch.cat([getattr(s, n).running_mean.clone() for s in singles]))
            self.register_buffer(n + "_rv", torch.cat([getattr(s, n).running_var.clone() for s in singles]))
        for n in ("f1", "f2"):
            setattr(self, n + "_weight", _lin_w(singles, lambda s, n=n: getattr(s, n).weight))
            setattr(self, n + "_bias", _lin_b(singles, lambda s, n=n: getattr(s, n).bias))

    def _bn(self, h, n):
        return F.batch_norm(h, getattr(self, n + "_rm"), getattr(self, n + "_rv"),
                            getattr(self, n + "_weight"), getattr(self, n + "_bias"),
                            self.training, 0.1, 1e-5)

    def forward(self, x, q):
        """x: (G, B, L); q: (G, B) -> (G, B)."""
        G, B, _ = x.shape
        h = x.permute(1, 0, 2).contiguous()
        for n, k in (("1", 0), ("2", 1), ("3", 2)):
            h = F.conv1d(h, getattr(self, "c" + n + "_weight"), getattr(self, "c" + n + "_bias"),
                         stride=2, groups=G)
            h = F.relu(self._bn(h, "b" + n))
        h = h.mean(-1).view(B, G, self.d).permute(1, 0, 2)
        h = torch.cat([h, q.unsqueeze(-1)], -1)
        h = F.relu(_lin(h, self.f1_weight, self.f1_bias))
        return _lin(h, self.f2_weight, self.f2_bias).squeeze(-1)

    def member_state(self, g: int) -> dict:
        """Parameters and BN buffers of member g, named like the single model's state_dict."""
        out = {}
        for n, ch in (("c1", self.a), ("c2", self.b), ("c3", self.d)):
            out[n + ".weight"] = getattr(self, n + "_weight")[g * ch:(g + 1) * ch]
            out[n + ".bias"] = getattr(self, n + "_bias")[g * ch:(g + 1) * ch]
        for n, ch in (("b1", self.a), ("b2", self.b), ("b3", self.d)):
            for k, suf in (("weight", "_weight"), ("bias", "_bias"), ("running_mean", "_rm"), ("running_var", "_rv")):
                out[n + "." + k] = getattr(self, n + suf)[g * ch:(g + 1) * ch]
        for n in ("f1", "f2"):
            out[n + ".weight"] = getattr(self, n + "_weight")[g].t()
            out[n + ".bias"] = getattr(self, n + "_bias")[g]
        return out


class BatchedM0(nn.Module):
    """G independent M0 members (einsum linear layers, weights (G, in, out))."""

    def __init__(self, singles):
        super().__init__()
        self.G = len(singles)
        self.register_buffer("idx", singles[0].idx.clone())
        for n in ("l1", "l2", "l3"):
            setattr(self, n + "_weight", _lin_w(singles, lambda s, n=n: getattr(s, n).weight))
            setattr(self, n + "_bias", _lin_b(singles, lambda s, n=n: getattr(s, n).bias))

    def forward(self, x, q):
        h = torch.cat([x[..., self.idx], q.unsqueeze(-1)], -1)
        h = F.relu(_lin(h, self.l1_weight, self.l1_bias))
        h = F.relu(_lin(h, self.l2_weight, self.l2_bias))
        return _lin(h, self.l3_weight, self.l3_bias).squeeze(-1)

    def member_state(self, g: int) -> dict:
        out = {}
        for n in ("l1", "l2", "l3"):
            out[n + ".weight"] = getattr(self, n + "_weight")[g].t()
            out[n + ".bias"] = getattr(self, n + "_bias")[g]
        return out


# --------------------------------------------------------------------------- ensemble wrapper
class Ensemble:
    """G members behind one interface. packing="batched": one Batched* module.
    packing="sequential": list of ordinary single models run one after another."""

    def __init__(self, kind: str, seeds, L: int, c: float = 1.0, packing: str = "batched", device="cuda"):
        assert kind in ("M1", "M0") and packing in ("batched", "sequential")
        self.kind, self.packing, self.G = kind, packing, len(seeds)
        singles = []
        for s in seeds:  # CPU init so every packing starts from identical weights
            torch.manual_seed(int(s))
            singles.append(M1(c) if kind == "M1" else M0(L))
        if packing == "batched":
            self.net = (BatchedM1 if kind == "M1" else BatchedM0)(singles).to(device)
            self.nets = None
        else:
            self.nets = nn.ModuleList(singles).to(device)
            self.net = None

    def parameters(self):
        return list((self.net if self.net is not None else self.nets).parameters())

    def train(self, mode: bool = True):
        (self.net if self.net is not None else self.nets).train(mode)

    def eval(self):
        self.train(False)

    def __call__(self, x, q):
        if self.net is not None:
            return self.net(x, q)
        return torch.stack([m(x[g], q[g]) for g, m in enumerate(self.nets)], 0)

    def clip_(self, clip: float):
        """Clip every member's gradient norm to `clip` (members stay independent)."""
        if self.net is not None:
            G, ps = self.G, [p for p in self.net.parameters() if p.grad is not None]
            sq = torch.zeros(G, device=ps[0].grad.device)
            for p in ps:
                sq += p.grad.reshape(G, -1).pow(2).sum(1)
            scale = (clip / (sq.sqrt() + 1e-6)).clamp(max=1.0)
            for p in ps:
                p.grad.view(G, -1).mul_(scale[:, None])
        else:
            for m in self.nets:
                torch.nn.utils.clip_grad_norm_(m.parameters(), clip)

    def member_state(self, g: int) -> dict:
        if self.net is not None:
            return self.net.member_state(g)
        return {k: v for k, v in self.nets[g].state_dict().items() if not k.endswith("num_batches_tracked") and k != "idx"}


# --------------------------------------------------------------------------- data + loss
class GData:
    """Tensors on the compute device. X (N, L) raw delta-Q; q0 (N,) Ah; y (N,) SOH in pp; cell (N,)."""

    def __init__(self, X, q0, y, cell, device="cuda"):
        self.X = torch.as_tensor(X, dtype=torch.float32, device=device)
        self.q0 = torch.as_tensor(q0, dtype=torch.float32, device=device)
        self.y = torch.as_tensor(y, dtype=torch.float32, device=device)
        self.cell = torch.as_tensor(cell, dtype=torch.long, device=device)
        self.cell_np = np.asarray(cell)
        self.L = int(self.X.shape[1])


def weighted_huber(pred, y, w, delta: float = 1.0):
    """Per-member cell-balanced weighted Huber. pred, y, w: (G, B) -> (G,)."""
    hub = F.huber_loss(pred, y, reduction="none", delta=delta)
    return (w * hub).sum(1) / w.sum(1).clamp_min(1e-12)


def make_train_rows(rows_by_cell: dict, stride: int, seed: int):
    """Every `stride`-th row per cell with a random per-cell offset; weights 1/(rows of that cell)
    so every cell has total weight 1. rows_by_cell: {cell_int: sorted global row indices}."""
    rows, w = [], []
    for cell, r in rows_by_cell.items():
        r = np.asarray(r)
        off = int(np.random.default_rng([int(seed), int(cell)]).integers(0, stride))
        sel = r[off::stride]
        if len(sel) == 0:
            sel = r[:1]
        rows.append(sel)
        w.append(np.full(len(sel), 1.0 / len(sel)))
    return np.concatenate(rows).astype(np.int64), np.concatenate(w).astype(np.float32)


def cell_macro(err: np.ndarray, cells: np.ndarray):
    """Cell-macro mean of abs errors; returns (macro, {cell: mae})."""
    u, inv = np.unique(cells, return_inverse=True)
    m = np.bincount(inv, weights=err) / np.bincount(inv)
    return float(m.mean()), {int(k): float(v) for k, v in zip(u, m)}


def compute_scalers(X_np, q0_np, y_np, cell_np, rows):
    """Fold-specific scalers from `rows` (all scored rows of the training cells) only.
    s: one scalar std of delta-Q (no per-point centering); q0: mean/sd over training cells;
    y: mean/sd over rows (pp)."""
    X, y = X_np[rows], y_np[rows]
    u, first = np.unique(cell_np[rows], return_index=True)
    q = q0_np[rows][first]
    return dict(s=float(X.std(dtype=np.float64)), q_mean=float(q.mean()), q_sd=float(q.std()),
                y_mean=float(y.mean()), y_sd=float(y.std()))


# --------------------------------------------------------------------------- trainer
def _member_tensors(members, dev):
    G = len(members)
    n = [len(m["train_rows"]) for m in members]
    maxn = max(n)
    TR = torch.zeros(G, maxn, dtype=torch.long)
    TW = torch.zeros(G, maxn)
    for g, m in enumerate(members):
        TR[g, :n[g]] = torch.as_tensor(m["train_rows"])
        TR[g, n[g]:] = int(m["train_rows"][0])
        TW[g, :n[g]] = torch.as_tensor(m["train_w"])
    f = lambda k: torch.tensor([m["scale"][k] for m in members], dtype=torch.float32, device=dev)
    sc = dict(s=f("s"), qm=f("q_mean"), qs=f("q_sd"), ym=f("y_mean"), ys=f("y_sd"))
    sc["res"] = torch.tensor([m["head"] == "res" for m in members], device=dev)
    return TR.to(dev), TW.to(dev), torch.tensor(n, device=dev), sc


def _base(gd, rows, sc, nominal):
    """Head offset in pp: abs -> train mean of y; res -> 100*q0/nominal (per row)."""
    return torch.where(sc["res"][:, None], 100.0 * gd.q0[rows] / nominal, sc["ym"][:, None].expand_as(rows).float())


def _inputs(gd, rows, sc):
    x = gd.X[rows] / sc["s"][:, None, None]
    q = (gd.q0[rows] - sc["qm"][:, None]) / sc["qs"][:, None]
    return x, q


class _EvalSet:
    def __init__(self, rows_list, gd, dev):
        self.G = len(rows_list)
        self.lens = [len(r) for r in rows_list]
        maxv = max(self.lens)
        R = torch.zeros(self.G, maxv, dtype=torch.long)
        for g, r in enumerate(rows_list):
            R[g, :len(r)] = torch.as_tensor(np.asarray(r))
        self.R = R.to(dev)
        self.rows = [np.asarray(r) for r in rows_list]
        self.cells = [gd.cell_np[r] for r in self.rows]


@torch.no_grad()
def predict(ens: Ensemble, gd: GData, es: _EvalSet, sc, nominal=NOMINAL_AH, chunk=2048):
    """Predictions in pp for each member's rows; returns list of numpy arrays."""
    ens.eval()
    out = torch.empty(es.G, es.R.shape[1], device=es.R.device)
    for st in range(0, es.R.shape[1], chunk):
        rows = es.R[:, st:st + chunk]
        x, q = _inputs(gd, rows, sc)
        z = ens(x, q)
        out[:, st:st + chunk] = _base(gd, rows, sc, nominal) + sc["ys"][:, None] * z
    out = out.cpu().numpy()
    return [out[g, :es.lens[g]] for g in range(es.G)]


def _score(preds, es, y_np):
    macro, per_cell = [], []
    for g in range(es.G):
        e = np.abs(preds[g] - y_np[es.rows[g]])
        m, d = cell_macro(e, es.cells[g])
        macro.append(m)
        per_cell.append(d)
    return np.array(macro), per_cell


def fit(kind, gd: GData, members, E: int, lr: float, *, c: float = 1.0, packing: str = "batched",
        ckpts=None, batch: int = 256, clip: float = 1.0, wd: float = 1e-4, delta: float = 1.0,
        noise: float = 0.02, seed: int = 42, nominal: float = NOMINAL_AH, return_preds: bool = False,
        device="cuda", eval_train: bool = True):
    """Train G independent members at once.

    members: list of dicts with train_rows, train_w, val_rows, seed, head ("abs"|"res"), scale
    (s, q_mean, q_sd, y_mean, y_sd). Each iteration every member draws `batch` rows from its own
    training rows (uniformly; loss weights are the cell-balanced row weights). One epoch is
    ceil(median training rows / batch) iterations. AdamW, cosine lr to 0 per iteration, Gaussian noise
    `noise` on the scaled input in training only, per-member gradient clipping, no early stopping.
    Evaluation at `ckpts` (epoch numbers) on ALL rows of each member's validation cells.
    """
    dev = torch.device(device)
    G = len(members)
    ens = Ensemble(kind, [m["seed"] for m in members], gd.L, c=c, packing=packing, device=dev)
    TR, TW, n_t, sc = _member_tensors(members, dev)
    ipe = int(math.ceil(float(np.median([len(m["train_rows"]) for m in members])) / batch))
    T = E * ipe
    ckpts = sorted(set(ckpts if ckpts is not None else [E]))
    opt = torch.optim.AdamW(ens.parameters(), lr=lr, weight_decay=wd)
    gen = torch.Generator(device=dev)
    gen.manual_seed(int(seed) * 7919 + int(sum(m["seed"] for m in members)) * 31 + G)
    es_val = _EvalSet([m["val_rows"] for m in members], gd, dev) if ckpts else None
    es_tr = _EvalSet([m["train_rows"] for m in members], gd, dev) if (ckpts and eval_train) else None
    y_np = gd.y.cpu().numpy()
    log, preds = [], None
    for it in range(T):
        cur = 0.5 * lr * (1.0 + math.cos(math.pi * it / T))
        for pg in opt.param_groups:
            pg["lr"] = cur
        ens.train()
        pos = (torch.rand(G, batch, device=dev, generator=gen) * n_t[:, None]).long().minimum(n_t[:, None] - 1)
        rows, w = TR.gather(1, pos), TW.gather(1, pos)
        x, q = _inputs(gd, rows, sc)
        if noise > 0:
            x = x + noise * torch.randn(x.shape, device=dev, generator=gen)
        z = ens(x, q)
        pred = _base(gd, rows, sc, nominal) + sc["ys"][:, None] * z
        loss = weighted_huber(pred, gd.y[rows], w, delta).sum()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        ens.clip_(clip)
        opt.step()
        done = (it + 1) // ipe if (it + 1) % ipe == 0 else None
        if done is not None and done in ckpts:
            pv = predict(ens, gd, es_val, sc, nominal)
            vm, vc = _score(pv, es_val, y_np)
            rec = dict(epoch=done, val_mae=vm, val_cell=vc)
            if es_tr is not None:
                rec["train_mae"], _ = _score(predict(ens, gd, es_tr, sc, nominal), es_tr, y_np)
            log.append(rec)
            preds = pv
    out = dict(log=log, ipe=ipe, iters=T, final=log[-1] if log else None)
    if return_preds:
        out["preds"] = preds
    out["ens"] = ens
    return out
'''
SD = SRC / "soh_deep.py"
SRC.mkdir(exist_ok=True)
if SD.exists():
    check("src/soh_deep.py already exists: not overwritten", True, f"sha256 {sha256_file(SD)[:12]}")
    check("existing soh_deep.py equals the version embedded in this notebook",
          SD.read_text(encoding="utf-8") == MODULE_SRC, "tests below run against the existing file", level="WARN")
else:
    SD.write_text(MODULE_SRC, encoding="utf-8")
    check("src/soh_deep.py created", True, f"{SD.stat().st_size} bytes")
import importlib, soh_deep as sd
importlib.reload(sd)

[PASS] src/soh_deep.py created: 18563 bytes


<module 'soh_deep' from 'c:\\workstation\\3RD YEAR\\Battery-Health-Estimation\\src\\soh_deep.py'>

In [5]:
# Tables, split-file hashes, development row set
def flat_hashes(o, out=None):
    out = {} if out is None else out
    if isinstance(o, dict):
        for k, v in o.items():
            if isinstance(v, str) and re.fullmatch(r"[0-9a-f]{64}", v): out[str(k)] = v
            else: flat_hashes(v, out)
    elif isinstance(o, list):
        for v in o:
            if isinstance(v, dict):
                nm = next((v[k] for k in ("path", "file", "name", "artifact") if k in v), None)
                hv = next((v[k] for k in ("sha256", "hash") if k in v and isinstance(v[k], str)), None)
                if nm and hv: out[str(nm)] = hv
                else: flat_hashes(v, out)
    return out
ART = flat_hashes(json.loads((P3 / "artifact_hashes.json").read_text()))
for fn in ["protocol_D.csv", "protocol_D_inner.csv"]:
    key = next((k for k in ART if Path(k).name == fn), None)
    if key is None:
        check(f"artifact hash for {fn}", False, "no entry found in artifact_hashes.json", level="WARN")
    else:
        check(f"{fn} matches artifact_hashes.json", sha256_file(SPL / fn) == ART[key], key)
KEPT_HASH = sha256_file(P3 / "kept_cells_groups_strata.csv")
print("sha256 kept_cells_groups_strata.csv:", KEPT_HASH)

LABELS = pd.read_parquet(PROC / "labels_table_v2.parquet")
CELLREF = pd.read_csv(PROC / "cell_reference.csv")
PD_ = pd.read_csv(SPL / "protocol_D.csv")
PI_ = pd.read_csv(SPL / "protocol_D_inner.csv")
KEPT = pd.read_csv(P3 / "kept_cells_groups_strata.csv")
ALLOW = json.loads((PROC / "feature_allowlist.json").read_text()) if (PROC / "feature_allowlist.json").exists() else {}
print("labels columns:", list(LABELS.columns)); print("protocol_D columns:", list(PD_.columns))
print("protocol_D_inner columns:", list(PI_.columns)); print("kept columns:", list(KEPT.columns))

c_cell = col(LABELS, ["cell_id", "cell", "cell_name"], "labels cell")
c_cyc = col(LABELS, ["cycle_number", "cycle", "cycle_idx"], "labels cycle")
c_sc2 = col(LABELS, ["scored2", "scored_2"], "labels scored2")
LAB = pd.DataFrame({"cell": LABELS[c_cell].astype(str).values, "cyc": LABELS[c_cyc].values,
                    "scored2": LABELS[c_sc2].astype(bool).values})
if "soh_nom" in LABELS.columns:
    LAB["y"] = 100.0 * LABELS["soh_nom"].values
else:
    LAB["y"] = 100.0 * LABELS[col(LABELS, ["qd_sum"], "labels qd_sum")].values / NOMINAL_AH
_lc = next((c for c in ["life", "cycle_life", "cell_life", "life_cycles"] if c in LABELS.columns), None)
LAB["life"] = LABELS[_lc].values if _lc else np.nan
check("life column found in labels", _lc is not None, _lc, level="WARN")
LAB["pos"] = np.arange(len(LAB))
q0_cell = dict(zip(CELLREF[col(CELLREF, ["cell_id", "cell", "cell_name"], "cell_reference cell")].astype(str),
                   CELLREF[col(CELLREF, ["q0"], "cell_reference q0")].astype(float)))
LAB["q0"] = LAB["cell"].map(q0_cell)

pc = col(PD_, ["cell_id", "cell", "cell_name"], "protocol_D cell")
DEV_CELLS = set(PD_[pc].astype(str))
DEV = LAB[LAB.scored2 & LAB.cell.isin(DEV_CELLS)].reset_index(drop=True)
DEV["dev_i"] = np.arange(len(DEV))
SENS = {"b1c8", "b1c10", "b1c12", "b1c13", "b1c22", "c8", "c10", "c12", "c13", "c22"}   # sensitivity cells
bad = [c for c in DEV_CELLS if c.lower().startswith("b3") or c.lower() in SENS]
check("no batch-3 or sensitivity cell in development cells", not bad, bad)
check("development rows == 55,963", len(DEV) == 55963, f"{len(DEV)} rows, {DEV.cell.nunique()} cells", level="WARN")
check("development cells == 84 (41 b1 + 43 b2)", len(DEV_CELLS) == 84, len(DEV_CELLS), level="WARN")
check("q0 available for all development cells", DEV.q0.notna().all(), int(DEV.q0.isna().sum()))

[PASS] protocol_D.csv matches artifact_hashes.json: data/processed/splits/protocol_D.csv
[PASS] protocol_D_inner.csv matches artifact_hashes.json: data/processed/splits/protocol_D_inner.csv
sha256 kept_cells_groups_strata.csv: c5f86b252d534aa2560d5c107af93529007f820deb3ff53dd7377fde408f29a0
labels columns: ['row_id', 'cell_id', 'batch', 'cycle_number', 'segment', 'in_life', 'qdlin_ok', 'causal_ok', 'label_suspect', 'post_junction', 'is_ref', 'q0', 'ref_last', 'score_start', 'soh_nom', 'soh_rel', 'soh_nom_repaired', 'scored', 'scored_incl_suspect', 'train_ok', 'ratio_end', 'max_step', 'qdlin_tail_suspect', 'qdlin_ok2', 'scored2', 'high_defect_cell']
protocol_D columns: ['repeat', 'outer_fold', 'cell_id', 'role', 'seed', 'tertile']
protocol_D_inner columns: ['repeat', 'outer_fold', 'inner_fold', 'cell_id', 'role']
kept columns: ['cell_id', 'batch', 'policy_norm', 'policy_str', 'life', 'position', 'is_dev', 'tertile', 'tertile_label']
[WARN] life column found in labels: None
[PASS] no bat

True

In [6]:
# Outer set (repeat 0, outer fold 0) and its 4 inner folds
def split_outer(P, r, o):
    rc = col(P, ["repeat", "rep", "repeat_id"], "repeat"); fc = col(P, ["outer_fold", "outer", "fold", "fold_id", "test_fold"], "outer fold")
    cc = col(P, ["cell_id", "cell", "cell_name"], "cell")
    role = next((c for c in ["role", "split", "set", "partition", "subset"] if c in P.columns), None)
    sub = P[P[rc] == r]
    if role is not None:
        sub = sub[sub[fc] == o]; is_te = sub[role].astype(str).str.lower().str.contains("test")
        return set(sub[~is_te][cc].astype(str)), set(sub[is_te][cc].astype(str))
    te = set(sub[sub[fc] == o][cc].astype(str)); return set(sub[cc].astype(str)) - te, te

_rc = col(PD_, ["repeat", "rep", "repeat_id"], "repeat"); _fc = col(PD_, ["outer_fold", "outer", "fold", "fold_id", "test_fold"], "outer fold")
R0, O0 = sorted(PD_[_rc].unique())[0], sorted(PD_[_fc].unique())[0]
OUTER_TRAIN, OUTER_TEST = split_outer(PD_, R0, O0)
print(f"repeat {R0}, outer fold {O0}: {len(OUTER_TRAIN)} outer-train cells; {len(OUTER_TEST)} outer-test cells (never loaded)")
check("outer train/test disjoint", not (OUTER_TRAIN & OUTER_TEST))
check("outer train U test == development cells", (OUTER_TRAIN | OUTER_TEST) == DEV_CELLS, "", level="WARN")

ic = col(PI_, ["cell_id", "cell", "cell_name"], "inner cell"); irc = col(PI_, ["repeat", "rep", "repeat_id"], "inner repeat")
ioc = col(PI_, ["outer_fold", "outer", "outer_id"], "inner outer"); iic = col(PI_, ["inner_fold", "inner", "inner_id", "val_fold", "fold"], "inner fold")
irole = next((c for c in ["role", "split", "set", "partition", "subset"] if c in PI_.columns), None)
SUBI = PI_[(PI_[irc] == R0) & (PI_[ioc] == O0)]
FOLDS = []
for k in sorted(SUBI[iic].unique()):
    s = SUBI[SUBI[iic] == k]
    if irole is not None:
        isval = s[irole].astype(str).str.lower().str.contains("val")
        vc = set(s[isval][ic].astype(str))
    else:
        vc = set(s[ic].astype(str))
    FOLDS.append(dict(inner=int(k), val_cells=sorted(vc), train_cells=sorted(OUTER_TRAIN - vc)))
check("4 inner folds", len(FOLDS) == 4, len(FOLDS))
for f in FOLDS:
    check(f"inner {f['inner']}: val subset of outer-train, disjoint from train, no outer-test cell",
          set(f["val_cells"]) <= OUTER_TRAIN and not (set(f["val_cells"]) & set(f["train_cells"])) and not (set(f["val_cells"] + f["train_cells"]) & OUTER_TEST),
          f"{len(f['train_cells'])} train / {len(f['val_cells'])} val cells")
check("inner validation folds partition the outer-train cells",
      sorted(sum([f["val_cells"] for f in FOLDS], [])) == sorted(OUTER_TRAIN), "", level="WARN")
# policy overlap (zero expected)
_kc = col(KEPT, ["cell_id", "cell", "cell_name"], "kept cell")
_gc = next((c for c in ["policy", "group", "policy_group", "charge_policy", "group_id"] if c in KEPT.columns), None)
if _gc:
    pol = dict(zip(KEPT[_kc].astype(str), KEPT[_gc].astype(str)))
    ov = [len({pol.get(c) for c in f["train_cells"]} & {pol.get(c) for c in f["val_cells"]}) for f in FOLDS]
    check("zero policy overlap between inner train and validation cells", sum(ov) == 0, ov, level="WARN")
else:
    check("policy column found in kept_cells_groups_strata.csv", False, list(KEPT.columns), level="WARN")

# working rows: outer-training cells only (outer-test rows never enter any array below)
WORK0 = DEV[DEV.cell.isin(OUTER_TRAIN)].sort_values(["cell", "cyc"]).reset_index(drop=True)
CODE = {c: i for i, c in enumerate(sorted(OUTER_TRAIN))}
CODE_INV = {v: k for k, v in CODE.items()}
check("working rows contain no outer-test cell", not (set(WORK0.cell) & OUTER_TEST), f"{len(WORK0)} rows, {WORK0.cell.nunique()} cells")
with guard("assert_allowed (soh_common) on working cells"):
    if hasattr(sc, "assert_allowed"):
        flex_call(sc.assert_allowed, dict(cells=sorted(OUTER_TRAIN), cell_ids=sorted(OUTER_TRAIN), ids=sorted(OUTER_TRAIN), cell_list=sorted(OUTER_TRAIN)))
        check("soh_common.assert_allowed accepted the working cells", True)

repeat 0, outer fold 0: 70 outer-train cells; 14 outer-test cells (never loaded)
[PASS] outer train/test disjoint: 
[PASS] outer train U test == development cells: 
[PASS] 4 inner folds: 4
[PASS] inner 0: val subset of outer-train, disjoint from train, no outer-test cell: 52 train / 18 val cells
[PASS] inner 1: val subset of outer-train, disjoint from train, no outer-test cell: 53 train / 17 val cells
[PASS] inner 2: val subset of outer-train, disjoint from train, no outer-test cell: 52 train / 18 val cells
[PASS] inner 3: val subset of outer-train, disjoint from train, no outer-test cell: 53 train / 17 val cells
[PASS] inner validation folds partition the outer-train cells: 
[WARN] policy column found in kept_cells_groups_strata.csv: ['cell_id', 'batch', 'policy_norm', 'policy_str', 'life', 'position', 'is_dev', 'tertile', 'tertile_label']
[PASS] working rows contain no outer-test cell: 43706 rows, 70 cells
[FAIL] assert_allowed (soh_common) on working cells: TypeError("assert_allowed

## 2. Data on GPU and scaling

Scalers are fold-specific and fitted on the inner-training cells only: one scalar `s` = std of all training delta-Q values (no per-point centering, so delta-Q[0] stays 0), q0 standardized by training mean/sd over cells, y centered by training mean/sd.
Heads: `abs` predicts `y_mean + y_sd*z` (pp); `res` predicts `100*q0/1.1 + y_sd*z` (pp), where `z` is the network output.

In [7]:
# Window widths from window_slice (fallback: grid_index) and delta-Q cache from Phase 4
def win_width(w):
    try:
        r = flex_call(sc.window_slice, dict(window=w, w=w, name=w, win=w, window_name=w))
        if isinstance(r, slice):
            return int((r.stop if r.stop is not None else 0) - (r.start or 0))
        if isinstance(r, (tuple, list)) and len(r) >= 2 and all(isinstance(v, (int, np.integer)) for v in r[:2]):
            return int(r[1] - r[0])
        if isinstance(r, (int, np.integer)):
            return int(r)
        raise TypeError(f"unexpected window_slice return {type(r)}")
    except Exception as e:
        ERRORS.append(dict(where=f"window_slice({w})", error=repr(e), trace=""))
        return int(round((3.5 - WINDOW_END_V[w]) / 0.0015015)) + 1
WIDTH = {w: win_width(w) for w in WINDOWS}
print("window widths:", WIDTH)

CACHE4 = OUT4 / "cache"
print("Phase 4 cache:")
for p in sorted(CACHE4.glob("*")):
    print(f"  {p.name:40s} {p.stat().st_size / 1e6:9.2f} MB")

def load_dq(w, work):
    pat = re.compile(r"(?<![A-Z0-9])" + w + r"(?![A-Z0-9])")
    files = [p for p in sorted(CACHE4.glob("*.npz")) if pat.search(p.stem.upper())]
    if len(files) != 1:
        raise FileNotFoundError(f"{w}: {len(files)} cache files match {[p.name for p in files]}")
    z = np.load(files[0], allow_pickle=False)
    arrs = {k: z[k] for k in z.files}
    print(f"  {files[0].name}: " + ", ".join(f"{k}{tuple(a.shape)}{a.dtype}" for k, a in arrs.items()))
    dqk = [k for k, a in arrs.items() if a.ndim == 2 and np.issubdtype(a.dtype, np.floating) and a.shape[1] == WIDTH[w]]
    if len(dqk) != 1:
        raise ValueError(f"{w}: expected one 2D float array of width {WIDTH[w]}; found {dqk}")
    full = arrs[dqk[0]]
    rowk = [k for k, a in arrs.items() if a.ndim == 1 and np.issubdtype(a.dtype, np.integer) and len(a) == len(full)]
    if rowk and set(work.pos) <= set(arrs[rowk[0]]):
        idx = pd.Index(arrs[rowk[0]]).get_indexer(work.pos.values)          # cache rows keyed by labels-table position
    elif len(full) == len(DEV):
        idx = work.dev_i.values                                              # cache rows in development-row order
    elif len(full) == len(LAB):
        idx = work.pos.values
    else:
        raise ValueError(f"{w}: cache rows {len(full)} match neither development rows ({len(DEV)}) nor labels ({len(LAB)})")
    X = np.ascontiguousarray(full[idx], dtype=np.float32)                    # slice to outer-train rows immediately
    del arrs, full, z
    return X

WDATA = {}
for w in PILOT_WINDOWS:
    with guard(f"load delta-Q {w}"):
        X = load_dq(w, WORK0)
        keep = np.isfinite(X).all(1)
        check(f"{w}: all working rows finite", keep.all(), f"{int((~keep).sum())} dropped", level="WARN")
        wk = WORK0[keep].reset_index(drop=True); X = X[keep]
        cell_code = wk.cell.map(CODE).values.astype(np.int64)
        gd = sd.GData(X, wk.q0.values, wk.y.values, cell_code, device=dev)
        rows_by_cell = {int(c): np.where(cell_code == c)[0] for c in np.unique(cell_code)}
        WDATA[w] = dict(gd=gd, X=X, q0=wk.q0.values.astype(np.float32), y=wk.y.values.astype(np.float32), cell=cell_code,
                     cyc=torch.as_tensor(wk.cyc.values, device=dev), life=torch.as_tensor(wk.life.values.astype(np.float32), device=dev),
                     rows_by_cell=rows_by_cell, work=wk)
        check(f"{w}: delta-Q[0] == 0 (no per-point centering)", float(np.abs(X[:, 0]).max()) < 1e-6, f"max |dq[0]| = {np.abs(X[:, 0]).max():.2e}", level="WARN")
        check(f"{w}: no outer-test cell in any array", not (set(wk.cell) & OUTER_TEST), f"X {X.shape}")
        print(w, "X", X.shape, "y range", float(wk.y.min()), float(wk.y.max()))

# recompute 3 random TRAINING cells with build_delta_q and compare with the cache (within 1e-6)
def recompute_check(w):
    rng = np.random.default_rng(SEED)
    cells = list(rng.choice(sorted(OUTER_TRAIN), 3, replace=False))
    wk = WDATA[w]["work"]; sel = wk.index[wk.cell.isin(cells)].values
    pos = wk.pos.values[sel]
    ctx = dict(window=w, w=w, name=w, win=w, window_name=w, cells=cells, cell_ids=cells, ids=cells, rows=pos, row_idx=pos, idx=pos,
               qdlin=np.load(PROC / "qdlin_f32.npy", mmap_mode="r"), qd=np.load(PROC / "qdlin_f32.npy", mmap_mode="r"),
               qref=np.load(PROC / "qref_f32.npy", mmap_mode="r"), qref_cell_ids=pd.read_csv(PROC / "qref_cell_ids.csv"),
               labels=LABELS, cycles=pd.read_parquet(PROC / "cycles_table.parquet"), cell_ref=CELLREF, cellref=CELLREF)
    r = flex_call(sc.build_delta_q, ctx)
    if isinstance(r, (tuple, list)):
        r = next(a for a in r if hasattr(a, "ndim") and a.ndim == 2)
    r = np.asarray(r, dtype=np.float32)
    ref = WDATA[w]["X"][sel]
    if r.shape == ref.shape:
        got = r
    elif r.shape[0] == len(LAB):
        got = r[pos]
    else:
        raise ValueError(f"build_delta_q returned {r.shape}; cache slice is {ref.shape}")
    d = float(np.nanmax(np.abs(got - ref)))
    check(f"{w}: 3-cell recompute equals Phase 4 cache within 1e-6", d < 1e-6, f"max abs diff {d:.2e} cells {cells}")
for w in WDATA:
    with guard(f"recompute check {w}"):
        recompute_check(w)

window widths: {'W310': 267, 'W305': 301, 'W300': 334, 'W280': 467, 'W240': 734, 'WFULL': 1000}
Phase 4 cache:
  dq_W240.npz                                 164.98 MB
  dq_W280.npz                                 105.21 MB
  dq_W300.npz                                  75.44 MB
  dq_W305.npz                                  68.05 MB
  dq_W310.npz                                  60.44 MB
  dq_WFULL.npz                                224.52 MB
  pred_B0a_ALL_r0_f0.parquet                    0.10 MB
  pred_B0a_ALL_r0_f1.parquet                    0.06 MB
  pred_B0a_ALL_r0_f2.parquet                    0.07 MB
  pred_B0a_ALL_r0_f3.parquet                    0.07 MB
  pred_B0a_ALL_r0_f4.parquet                    0.08 MB
  pred_B0a_ALL_r0_f5.parquet                    0.06 MB
  pred_B0a_ALL_r1_f0.parquet                    0.10 MB
  pred_B0a_ALL_r1_f1.parquet                    0.09 MB
  pred_B0a_ALL_r1_f2.parquet                    0.06 MB
  pred_B0a_ALL_r1_f3.parquet                    0

c:\workstation\3RD YEAR\Battery-Health-Estimation\src\soh_deep.py:238: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  self.q0 = torch.as_tensor(q0, dtype=torch.float32, device=device)


[PASS] W300: no outer-test cell in any array: X (43706, 334)
W300 X (43706, 334) y range 79.47254727272727 99.8484909090909
  dq_W240.npz: qidx(55963,)int64, dq(55963, 734)float32, w(55963,)float32
[PASS] W240: all working rows finite: 0 dropped
[PASS] W240: delta-Q[0] == 0 (no per-point centering): max |dq[0]| = 0.00e+00
[PASS] W240: no outer-test cell in any array: X (43706, 734)
W240 X (43706, 734) y range 79.47254727272727 99.8484909090909
[FAIL] recompute check W300: TypeError("build_delta_q(qdlin_row, qref, i_open, i_close, n_points=None): unmapped required parameters ['qdlin_row', 'i_open', 'i_close']; ctx keys ['cell_ids', 'cell_ref', 'cellref', 'cells', 'cycles', 'ids', 'idx', 'labels', 'name', 'qd', 'qdlin', 'qref', 'qref_cell_ids', 'row_idx', 'rows', 'w', 'win', 'window', 'window_name']")
[FAIL] recompute check W240: TypeError("build_delta_q(qdlin_row, qref, i_open, i_close, n_points=None): unmapped required parameters ['qdlin_row', 'i_open', 'i_close']; ctx keys ['cell_ids'

In [8]:
# Members = the 4 inner folds; fold-specific scalers from training cells only
def make_members(w, head, seed_off=0, folds=FOLDS):
    d = WDATA[w]; out = []
    for f in folds:
        tr_codes = [CODE[c] for c in f["train_cells"]]; va_codes = [CODE[c] for c in f["val_cells"]]
        rbc = {c: d["rows_by_cell"][c] for c in tr_codes}
        tr_rows, tr_w = sd.make_train_rows(rbc, TRAIN_STRIDE, SEED)
        all_tr = np.concatenate([d["rows_by_cell"][c] for c in tr_codes])
        sc_ = sd.compute_scalers(d["X"], d["q0"], d["y"], d["cell"], all_tr)
        va_rows = np.concatenate([d["rows_by_cell"][c] for c in va_codes])
        out.append(dict(inner=f["inner"], train_rows=tr_rows, train_w=tr_w, val_rows=va_rows, seed=SEED + seed_off + f["inner"],
                        head=head, scale=sc_, train_cells=f["train_cells"], val_cells=f["val_cells"], all_train_rows=all_tr))
    return out

for w in WDATA:
    with guard(f"scalers {w}"):
        ms = make_members(w, "abs"); d = WDATA[w]
        for m in ms:
            # independent recompute: concatenate per-cell blocks of the training cells only
            Xc = np.concatenate([d["X"][d["work"].index[d["work"].cell == c].values] for c in m["train_cells"]])
            yc = np.concatenate([d["y"][d["work"].index[d["work"].cell == c].values] for c in m["train_cells"]])
            qc = np.array([d["work"].q0[d["work"].cell == c].iloc[0] for c in m["train_cells"]])
            ok = (abs(Xc.std(dtype=np.float64) - m["scale"]["s"]) < 1e-6 * max(1, m["scale"]["s"]) and abs(yc.mean() - m["scale"]["y_mean"]) < 1e-4
                  and abs(yc.std() - m["scale"]["y_sd"]) < 1e-4 and abs(qc.mean() - m["scale"]["q_mean"]) < 1e-6 and abs(qc.std() - m["scale"]["q_sd"]) < 1e-6)
            check(f"{w} inner {m['inner']}: scalers equal recomputed training-cell values", ok, {k: round(v, 5) for k, v in m["scale"].items()})

[PASS] W300 inner 0: scalers equal recomputed training-cell values: {'s': 0.06829, 'q_mean': 1.07803, 'q_sd': 0.00727, 'y_mean': 94.24657, 'y_sd': 4.63723}
[PASS] W300 inner 1: scalers equal recomputed training-cell values: {'s': 0.06815, 'q_mean': 1.07762, 'q_sd': 0.00838, 'y_mean': 94.18964, 'y_sd': 4.62967}
[PASS] W300 inner 2: scalers equal recomputed training-cell values: {'s': 0.06855, 'q_mean': 1.07778, 'q_sd': 0.00885, 'y_mean': 94.34645, 'y_sd': 4.62032}
[PASS] W300 inner 3: scalers equal recomputed training-cell values: {'s': 0.06872, 'q_mean': 1.07771, 'q_sd': 0.00767, 'y_mean': 94.15233, 'y_sd': 4.66161}
[PASS] W240 inner 0: scalers equal recomputed training-cell values: {'s': 0.08322, 'q_mean': 1.07803, 'q_sd': 0.00727, 'y_mean': 94.24657, 'y_sd': 4.63723}
[PASS] W240 inner 1: scalers equal recomputed training-cell values: {'s': 0.08259, 'q_mean': 1.07762, 'q_sd': 0.00838, 'y_mean': 94.18964, 'y_sd': 4.62967}
[PASS] W240 inner 2: scalers equal recomputed training-cell valu

## 3. Model code

`src/soh_deep.py` (written above): `M1` (2,977 parameters), `M0` (about 2,969), `BatchedM1`/`BatchedM0` (grouped conv, per-member BatchNorm channels, einsum linear layers, stacked initialisation from ordinary single models, per-member gradient clipping), `Ensemble` (batched or sequential packing) and `fit` (parallel trainer: per-member rows, cell weights, seed and head; cell-balanced weighted Huber; AdamW; per-iteration cosine lr; noise on scaled input; no early stopping; no model file is ever read or written).

## 4. Tests

In [9]:
# 4a. parameter counts
L3 = WIDTH["W300"]
cnt = {c: sd.n_params(sd.M1(c)) for c in (0.5, 1, 2, 4)}
print("M1 parameter counts:", cnt, "| ratios vs c=1:", {c: round(v / cnt[1], 2) for c, v in cnt.items()})
check("4a M1(c=1) has exactly 2,977 parameters", cnt[1] == 2977, cnt[1])
check("4a size ratios about 0.3 / 1 / 3.7 / 15 (D36)", all(abs(cnt[c] / cnt[1] - r) / r < 0.15 for c, r in ((0.5, 0.3), (2, 3.7), (4, 15))), {c: round(v / cnt[1], 2) for c, v in cnt.items()}, level="WARN")
n_m0 = sd.n_params(sd.M0(L3))
print("M0 parameters:", n_m0)
check("4a M0 parameter count close to M1", abs(n_m0 - cnt[1]) / cnt[1] < 0.05, n_m0, level="WARN")

# 4b. batched vs single parity (eval mode at init, weights after 20 identical steps)
def fixed_batches(G, B, L, n, seed=0):
    g = torch.Generator().manual_seed(seed); out = []
    for _ in range(n):
        out.append((torch.randn(G, B, L, generator=g) * 0.5, torch.randn(G, B, generator=g),
                    torch.randn(G, B, generator=g) * 3.0, torch.rand(G, B, generator=g) + 0.5))
    return out

def train_steps(ens, batches, lr=1e-3):
    opt = torch.optim.AdamW(ens.parameters(), lr=lr, weight_decay=WD)
    ens.train()
    for x, q, y, w in batches:
        x, q, y, w = [t.to(dev) for t in (x, q, y, w)]
        loss = sd.weighted_huber(ens(x, q), y, w, HUBER_DELTA).sum()
        opt.zero_grad(set_to_none=True); loss.backward(); ens.clip_(CLIP); opt.step()

@torch.no_grad()
def eval_out(ens, x, q):
    ens.eval(); return ens(x.to(dev), q.to(dev)).cpu()

def state_diff(ea, eb):
    worst = 0.0
    for g in range(ea.G):
        sa, sb = ea.member_state(g), eb.member_state(g)
        assert set(sa) == set(sb), (set(sa) ^ set(sb))
        for k in sa:
            worst = max(worst, (sa[k].detach() - sb[k].detach()).abs().max().item())
    return worst

for kind in ("M1", "M0"):
    with guard(f"4b parity {kind}"):
        seeds = [11, 12, 13]
        eb = sd.Ensemble(kind, seeds, L3, packing="batched", device=dev)
        es = sd.Ensemble(kind, seeds, L3, packing="sequential", device=dev)
        xt, qt = torch.randn(3, 64, L3) * 0.5, torch.randn(3, 64)
        d0 = (eval_out(eb, xt, qt) - eval_out(es, xt, qt)).abs().max().item()
        check(f"4b {kind}: batched == single models in eval mode (init)", d0 < 1e-5, f"max abs diff {d0:.2e}")
        bt = fixed_batches(3, 64, L3, 20)
        train_steps(eb, bt); train_steps(es, bt)
        dw = state_diff(eb, es)
        d1 = (eval_out(eb, xt, qt) - eval_out(es, xt, qt)).abs().max().item()
        check(f"4b {kind}: weights equal after 20 identical steps", dw < 1e-4, f"max abs weight diff {dw:.2e}; eval output diff {d1:.2e}")

# 4c. overfit test on 256 fixed rows
with guard("4c overfit test"):
    d = WDATA["W300"]; m0 = make_members("W300", "abs")[0]; gd = d["gd"]; s_ = m0["scale"]
    rows = np.random.default_rng(SEED).choice(m0["train_rows"], 256, replace=False)
    rt = torch.as_tensor(rows, device=dev)
    x = (gd.X[rt] / s_["s"])[None]; q = ((gd.q0[rt] - s_["q_mean"]) / s_["q_sd"])[None]; y = gd.y[rt]
    torch.manual_seed(SEED)
    ens = sd.Ensemble("M1", [SEED], gd.L, packing="sequential", device=dev)
    OV_LR, OV_STEPS = 1e-2, 800
    opt = torch.optim.AdamW(ens.parameters(), lr=OV_LR, weight_decay=0.0)
    best, hist = 1e9, []
    for step in range(OV_STEPS):
        for pg in opt.param_groups: pg["lr"] = 0.5 * OV_LR * (1 + math.cos(math.pi * step / OV_STEPS))
        ens.train()
        pred = s_["y_mean"] + s_["y_sd"] * ens(x, q)[0]
        loss = torch.nn.functional.huber_loss(pred, y, delta=HUBER_DELTA)
        opt.zero_grad(set_to_none=True); loss.backward(); ens.clip_(CLIP); opt.step()
        if (step + 1) % 25 == 0:
            ens.eval()
            with torch.no_grad():
                mae = (s_["y_mean"] + s_["y_sd"] * ens(x, q)[0] - y).abs().mean().item()
            best = min(best, mae); hist.append((step + 1, mae))
    check("4c M1 overfits 256 rows: train MAE < 0.15 pp within 800 steps", best < 0.15, f"best eval-mode train MAE {best:.4f} pp (lr {OV_LR}); last {hist[-1]}")

# 4d. repeatability + 4e. separation (uses the same two runs)
with guard("4d repeatability / 4e separation"):
    ms = make_members("W300", "abs")[:2]; gd = WDATA["W300"]["gd"]
    r1 = sd.fit("M1", gd, ms, 2, 2e-3, ckpts=[2], return_preds=True, noise=NOISE_SIGMA, seed=SEED, packing="batched", device=dev)
    r2 = sd.fit("M1", gd, ms, 2, 2e-3, ckpts=[2], return_preds=True, noise=NOISE_SIGMA, seed=SEED, packing="batched", device=dev)
    same = all(np.array_equal(a, b) for a, b in zip(r1["preds"], r2["preds"]))
    check("4d same seed gives identical predictions (exact)", same, f"max abs diff {max(np.abs(a - b).max() for a, b in zip(r1['preds'], r2['preds'])):.2e}")
    check("4e predictions finite", all(np.isfinite(p).all() for p in r1["preds"]))
for w in WDATA:
    with guard(f"4e separation {w}"):
        gd = WDATA[w]["gd"]; ok = True; det = []
        for hd in ("abs", "res"):
            for m in make_members(w, hd):
                trc = {CODE_INV[c] for c in np.unique(gd.cell_np[m["train_rows"]])}; vac = {CODE_INV[c] for c in np.unique(gd.cell_np[m["val_rows"]])}
                good = (not (trc & vac)) and trc <= set(m["train_cells"]) <= OUTER_TRAIN and vac <= set(m["val_cells"]) and not ((trc | vac) & OUTER_TEST)
                ok &= good; det.append(good)
        check(f"4e {w}: val cells disjoint from train cells, train cells are outer-train (r{R0},o{O0}) only", ok, det)

# 4f. cell weights
with guard("4f cell weights"):
    worst = 0.0
    for w in WDATA:
        gd = WDATA[w]["gd"]
        for m in make_members(w, "abs"):
            cc = gd.cell_np[m["train_rows"]]
            sums = np.array([m["train_w"][cc == c].sum() for c in np.unique(cc)], dtype=np.float64)
            worst = max(worst, float(sums.max() - sums.min()))
    check("4f weight sum per training cell equal within 1e-6", worst < 1e-6, f"max spread {worst:.2e}")

# 4g. text scan (patterns built by concatenation so this cell does not match itself)
BANNED = ["torch" + ".load", "load_state" + "_dict", "from_" + "pretrained", "torch" + "vision", "torch" + "audio"]
def scan(path):
    t = Path(path).read_text(encoding="utf-8", errors="ignore")
    return [b for b in BANNED if b in t]
hits = {"src/soh_deep.py": scan(SD)}
nb_candidates = [Path.cwd() / "05a_deep_pilot_gpu.ipynb", ROOT / "notebooks/05a_deep_pilot_gpu.ipynb", ROOT / "05a_deep_pilot_gpu.ipynb"]
nb_path = next((p for p in nb_candidates if p.exists()), None)
if nb_path is not None:
    hits["notebook"] = scan(nb_path)
check("4g no pretrained-weight or vision/audio-library use in module and notebook", not any(hits.values()) and nb_path is not None,
      hits if nb_path is not None else f"notebook file not found next to cwd; module hits {hits}", level="FAIL" if nb_path is not None else "WARN")

M1 parameter counts: {0.5: 817, 1: 2977, 2: 11329, 4: 44161} | ratios vs c=1: {0.5: 0.27, 1: 1.0, 2: 3.81, 4: 14.83}
[PASS] 4a M1(c=1) has exactly 2,977 parameters: 2977
[PASS] 4a size ratios about 0.3 / 1 / 3.7 / 15 (D36): {0.5: 0.27, 1: 1.0, 2: 3.81, 4: 14.83}
M0 parameters: 2969
[PASS] 4a M0 parameter count close to M1: 2969
[PASS] 4b M1: batched == single models in eval mode (init): max abs diff 1.49e-08
[FAIL] 4b M1: weights equal after 20 identical steps: max abs weight diff 1.43e-03; eval output diff 1.13e-04
[PASS] 4b M0: batched == single models in eval mode (init): max abs diff 2.24e-08
[PASS] 4b M0: weights equal after 20 identical steps: max abs weight diff 4.00e-08; eval output diff 2.24e-08
[FAIL] 4c M1 overfits 256 rows: train MAE < 0.15 pp within 800 steps: best eval-mode train MAE 0.6399 pp (lr 0.01); last (800, 0.6436774730682373)
[PASS] 4d same seed gives identical predictions (exact): max abs diff 0.00e+00
[PASS] 4e predictions finite: 
[PASS] 4e W300: val cells dis

True

## 5. Benchmark

Synthetic inputs of the real window widths (speed does not depend on values). An epoch is 28 iterations (7,000 training rows per member, batch 256). Sequential packing runs ordinary single models one after another inside each iteration; batched packing runs one grouped-conv model.

In [10]:
BENCH_PATH = OUT / "benchmark.csv"
N_SYN, ROWS_PER_MEMBER = 8000, 7000
def syn_gd(L):
    rng = np.random.default_rng(0)
    return sd.GData(rng.normal(0, 0.05, (N_SYN, L)).astype(np.float32), rng.normal(1.08, 0.01, N_SYN), rng.normal(90, 5, N_SYN),
                    rng.integers(0, 60, N_SYN), device=dev)
def syn_members(G):
    rng = np.random.default_rng(1); out = []
    for g in range(G):
        tr = rng.choice(N_SYN, ROWS_PER_MEMBER, replace=False)
        out.append(dict(train_rows=tr, train_w=np.full(ROWS_PER_MEMBER, 1.0 / ROWS_PER_MEMBER, np.float32), val_rows=tr[:10], seed=SEED + g,
                        head="abs", scale=dict(s=0.05, q_mean=1.08, q_sd=0.01, y_mean=90.0, y_sd=5.0)))
    return out

def bench_one(kind, c, G, packing, gd):
    # per-epoch time from the difference of a 3-epoch and a 1-epoch run (model construction cancels)
    try:
        ms = syn_members(G)
        sd.fit(kind, gd, ms, 1, 1e-3, c=c, packing=packing, ckpts=[], noise=NOISE_SIGMA, device=dev)       # warm-up
        ts = {}
        for E in (1, 3):
            torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t = time.perf_counter()
            sd.fit(kind, gd, ms, E, 1e-3, c=c, packing=packing, ckpts=[], noise=NOISE_SIGMA, device=dev)
            torch.cuda.synchronize(); ts[E] = time.perf_counter() - t
        peak = torch.cuda.max_memory_allocated() / 1e9
        dt = max((ts[3] - ts[1]) / 2, 1e-6)
        return dict(sec_per_epoch=dt, ms_per_iter=1000 * dt / 28, epochs_per_s_member=1 / dt, member_epochs_per_s=G / dt, peak_mem_gb=peak)
    except torch.cuda.OutOfMemoryError:
        ERRORS.append(dict(where=f"bench {kind} c={c} G={G} {packing}", error="CUDA OOM", trace=""))
        return dict(sec_per_epoch=np.nan, ms_per_iter=np.nan, epochs_per_s_member=np.nan, member_epochs_per_s=np.nan, peak_mem_gb=np.nan)
    finally:
        gc.collect(); torch.cuda.empty_cache()

G_LIST = [1, 4, 8, 18, 36, 72]
if BENCH_PATH.exists() and (CACHE / f"bench_{CODE_VERSION}.ok").exists():
    BENCH = pd.read_csv(BENCH_PATH); print("benchmark reloaded from", BENCH_PATH)
else:
    rows = []
    for w in WINDOWS:
        with guard(f"benchmark {w}"):
            gd = syn_gd(WIDTH[w])
            rows.append(dict(window=w, width=WIDTH[w], model="M1", c=1.0, packing="sequential", G=4, **bench_one("M1", 1.0, 4, "sequential", gd)))
            for G in G_LIST:
                rows.append(dict(window=w, width=WIDTH[w], model="M1", c=1.0, packing="batched", G=G, **bench_one("M1", 1.0, G, "batched", gd)))
            if w == "W300":
                rows.append(dict(window=w, width=WIDTH[w], model="M1", c=4.0, packing="batched", G=4, **bench_one("M1", 4.0, 4, "batched", gd)))
                rows.append(dict(window=w, width=WIDTH[w], model="M0", c=1.0, packing="batched", G=72, **bench_one("M0", 1.0, 72, "batched", gd)))
            del gd; gc.collect(); torch.cuda.empty_cache()
        print(w, "done", f"{time.time() - T_START:.0f}s elapsed")
    BENCH = pd.DataFrame(rows); BENCH.to_csv(BENCH_PATH, index=False); (CACHE / f"bench_{CODE_VERSION}.ok").write_text("ok")
print(BENCH.round(3).to_string(index=False))

# packing choice: highest members x epochs per second, averaged over the two pilot windows (M1, c=1)
bt = BENCH[(BENCH.model == "M1") & (BENCH.c == 1.0) & BENCH.window.isin(PILOT_WINDOWS)]
agg = bt.groupby(["packing", "G"]).member_epochs_per_s.mean().dropna()
best_pack, best_G = agg.idxmax()
seq_thr = float(agg.get(("sequential", 4), np.nan))
PACK = dict(packing=best_pack, G=int(best_G), member_epochs_per_s=float(agg.max()), sequential_member_epochs_per_s=seq_thr)
slower = [(int(g), round(float(v), 2)) for (p, g), v in agg.items() if p == "batched" and v < seq_thr]
print("members x epochs/s (mean of pilot windows):", {f"{p}-G{g}": round(float(v), 2) for (p, g), v in agg.items()})
print(f"Chosen packing: {best_pack} (G={best_G}).",
      ("Grouped conv is SLOWER than sequential single models at G = " + str([g for g, _ in slower])) if slower else "Grouped conv beats sequential at every G tested.")
PILOT_PACKING = "batched" if float(agg.get(("batched", 8), 0)) >= seq_thr else "sequential"
print("Packing used in the pilot (8 members = 4 inner folds x 2 heads):", PILOT_PACKING)

W310 done 89s elapsed
W305 done 126s elapsed
W300 done 171s elapsed
W280 done 218s elapsed
W240 done 284s elapsed
WFULL done 369s elapsed
window  width model   c    packing  G  sec_per_epoch  ms_per_iter  epochs_per_s_member  member_epochs_per_s  peak_mem_gb
  W310    267    M1 1.0 sequential  4          1.123       40.118                0.890                3.561        0.295
  W310    267    M1 1.0    batched  1          0.486       17.356                2.058                2.058        0.275
  W310    267    M1 1.0    batched  4          0.367       13.123                2.721               10.886        0.302
  W310    267    M1 1.0    batched  8          0.587       20.970                1.703               13.625        0.337
  W310    267    M1 1.0    batched 18          0.600       21.445                1.665               29.978        0.429
  W310    267    M1 1.0    batched 36          1.264       45.128                0.791               28.490        0.589
  W310    267  

## 6. Pilot on inner folds only (repeat 0, outer fold 0)

Members are the 4 inner folds (train on inner-train cells, validate on inner-validation cells). Both heads are trained in the same run (8 members). A fresh run per (E, lr) so each has its own cosine schedule. Final-epoch MAE is reported (no early stopping, no checkpoint selection). Results are cached per config, so the notebook is resumable.
Tree comparators are tuned on the same inner folds they are scored on, which favours the baselines slightly (conservative for the M1 comparison).

In [11]:
PILOT_E, PILOT_LR = [30, 60, 120], [1e-3, 2e-3, 4e-3]
NF = len(FOLDS)
def ck(E):
    return sorted({max(1, int(round(E * f))) for f in (1 / 6, 2 / 6, 3 / 6, 4 / 6, 5 / 6, 1.0)})
def cached(key, fn):
    p = CACHE / f"{CODE_VERSION}_{key}.pkl"
    if p.exists():
        with open(p, "rb") as f:
            return pickle.load(f)
    r = fn()
    with open(p, "wb") as f:
        pickle.dump(r, f)
    return r

def run_cfg(w, kind, E, lr, heads=("abs", "res"), c=1.0, seed_off=0):
    key = f"{w}_{kind}_c{c:g}_E{E}_lr{lr:g}_{'-'.join(heads)}_s{seed_off}"
    def go():
        ms = sum([make_members(w, h, seed_off) for h in heads], [])
        t = time.time()
        r = sd.fit(kind, WDATA[w]["gd"], ms, E, lr, c=c, packing=PILOT_PACKING, ckpts=ck(E), noise=NOISE_SIGMA, seed=SEED + seed_off, device=dev)
        out = dict(secs=time.time() - t, ipe=r["ipe"], heads=list(heads), E=E, lr=lr, kind=kind, c=c, window=w, seed_off=seed_off,
                   n_params=sd.n_params(sd.M1(c) if kind == "M1" else sd.M0(WIDTH[w])), ckpts=[])
        for rec in r["log"]:
            out["ckpts"].append(dict(epoch=rec["epoch"], val=rec["val_mae"].reshape(len(heads), NF), train=rec["train_mae"].reshape(len(heads), NF)))
        del r; gc.collect(); torch.cuda.empty_cache()
        return out
    return cached(key, go)

REC = []
for w in PILOT_WINDOWS:
    for kind in ("M1", "M0"):
        for E, lr in itertools.product(PILOT_E, PILOT_LR):
            with guard(f"pilot {w} {kind} E={E} lr={lr}"):
                o = run_cfg(w, kind, E, lr)
                last = o["ckpts"][-1]
                for hi, h in enumerate(o["heads"]):
                    REC.append(dict(window=w, model=kind, head=h, E=E, lr=lr, val_mae=float(last["val"][hi].mean()), train_mae=float(last["train"][hi].mean()),
                                    gap=float(last["val"][hi].mean() - last["train"][hi].mean()), val_fold_sd=float(last["val"][hi].std(ddof=1)), secs=o["secs"]))
        print(f"{w} {kind} grid done, {time.time() - T_START:.0f}s elapsed")
PILOT = pd.DataFrame(REC); PILOT.to_csv(OUT / "pilot_log.csv", index=False)
print(PILOT.groupby(["window", "model", "head"]).val_mae.min().round(4).to_string())

W300 M1 grid done, 620s elapsed
W300 M0 grid done, 736s elapsed
W240 M1 grid done, 1227s elapsed
W240 M0 grid done, 1348s elapsed
window  model  head
W240    M0     abs     0.5680
               res     0.5507
        M1     abs     0.6849
               res     0.6687
W300    M0     abs     0.8632
               res     0.8435
        M1     abs     0.8300
               res     0.8347


In [12]:
# Best configs and extra experiments: (i) seeds, (ii) size sweep
def best_row(w, kind, head=None):
    s = PILOT[(PILOT.window == w) & (PILOT.model == kind)]
    if head: s = s[s["head"] == head]
    return s.loc[s.val_mae.idxmin()]

SEEDS = {}
for w in PILOT_WINDOWS:
    with guard(f"seed sd {w}"):
        b = best_row(w, "M1"); maes = [float(b.val_mae)]
        for j in (1, 2, 3):
            o = run_cfg(w, "M1", int(b.E), float(b.lr), heads=(b["head"],), seed_off=1000 * j)
            maes.append(float(o["ckpts"][-1]["val"][0].mean()))
        SEEDS[w] = dict(head=b["head"], E=int(b.E), lr=float(b.lr), maes=maes, mean=float(np.mean(maes)), sd=float(np.std(maes, ddof=1)))
        print(w, SEEDS[w])

SIZE = {}
with guard("size sweep W300"):
    b = best_row("W300", "M1", None)
    for c in (0.5, 1.0, 2.0, 4.0):
        if c == 1.0:
            SIZE[c] = dict(n_params=2977, mae=float(b.val_mae), head=b["head"], E=int(b.E), lr=float(b.lr))   # c=1 result comes from the grid
        else:
            o = run_cfg("W300", "M1", int(b.E), float(b.lr), heads=(b["head"],), c=c)
            SIZE[c] = dict(n_params=o["n_params"], mae=float(o["ckpts"][-1]["val"][0].mean()), head=b["head"], E=int(b.E), lr=float(b.lr), gap=float((o["ckpts"][-1]["val"][0] - o["ckpts"][-1]["train"][0]).mean()))
    print(pd.DataFrame(SIZE).T.to_string())

W300 {'head': 'abs', 'E': 120, 'lr': 0.004, 'maes': [0.830011554930721, 0.8034802507711597, 0.8337122411239959, 0.8652498904405062], 'mean': 0.8331134843165956, 'sd': 0.02530387061662454}
W240 {'head': 'res', 'E': 120, 'lr': 0.001, 'maes': [0.6686827646804892, 0.7481159286115041, 0.8102412550586513, 0.7897037813251464], 'mean': 0.7541859324189477, 'sd': 0.06258688208322408}
    n_params       mae head    E     lr       gap
0.5      817   0.78936  abs  120  0.004  0.073703
1.0     2977  0.830012  abs  120  0.004       NaN
2.0    11329  0.848545  abs  120  0.004   0.14891
4.0    44161  0.839147  abs  120  0.004  0.174942


In [13]:
# (iii) same-fold comparators on the same inner folds and training rows
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
def feats(w, rows):
    d = WDATA[w]; idx = sd.sub_idx(d["gd"].L)
    return np.column_stack([d["X"][rows][:, idx], d["q0"][rows]])

def comp_eval(w, fam, par):
    # B0b: weighted linear regression on q0 only (ASSUMED definition; compare with Phase 4 baseline_results.csv).
    # B3/B3r: gradient boosting, 150 trees. B4/B4r: random forest, 150 trees. *r: target = soh_nom - q0/1.1 (pp: y - 100*q0/1.1).
    d = WDATA[w]; resid = fam in ("B3r", "B4r"); maes = []; t = time.time()
    for m in make_members(w, "abs"):
        tr, va = m["train_rows"], m["val_rows"]
        off_tr = 100.0 * d["q0"][tr] / NOMINAL_AH if resid else 0.0
        off_va = 100.0 * d["q0"][va] / NOMINAL_AH if resid else 0.0
        if fam == "B0b":
            mdl = LinearRegression().fit(d["q0"][tr][:, None], d["y"][tr], sample_weight=m["train_w"]); pred = mdl.predict(d["q0"][va][:, None])
        elif fam in ("B3", "B3r"):
            mdl = GradientBoostingRegressor(n_estimators=150, max_depth=par["depth"], learning_rate=par["lr"], random_state=SEED)
            mdl.fit(feats(w, tr), d["y"][tr] - off_tr, sample_weight=m["train_w"]); pred = mdl.predict(feats(w, va)) + off_va
        else:
            mdl = RandomForestRegressor(n_estimators=150, min_samples_leaf=par["leaf"], max_features=1 / 3, n_jobs=-1, random_state=SEED)
            mdl.fit(feats(w, tr), d["y"][tr] - off_tr, sample_weight=m["train_w"]); pred = mdl.predict(feats(w, va)) + off_va
        maes.append(sd.cell_macro(np.abs(pred - d["y"][va]), d["cell"][va])[0])
    return dict(maes=maes, mae=float(np.mean(maes)), secs=time.time() - t)

COMP_GRID = {"B0b": [{}],
             "B3": [dict(depth=a, lr=b) for a in (2, 3) for b in (0.05, 0.1)], "B3r": [dict(depth=a, lr=b) for a in (2, 3) for b in (0.05, 0.1)],
             "B4": [dict(leaf=l) for l in (5, 10, 20)], "B4r": [dict(leaf=l) for l in (1, 2, 5)]}
CR = []
for w in PILOT_WINDOWS:
    for fam, grid in COMP_GRID.items():
        for par in grid:
            with guard(f"comparator {w} {fam} {par}"):
                r = cached(f"comp_{w}_{fam}_{'_'.join(f'{k}{v}' for k, v in par.items())}", lambda: comp_eval(w, fam, par))
                CR.append(dict(window=w, family=fam, params=json.dumps(par), mae=r["mae"], secs=r["secs"], secs_per_fit=r["secs"] / NF))
    print(w, "comparators done", f"{time.time() - T_START:.0f}s elapsed")
COMP = pd.DataFrame(CR)
COMP_BEST = COMP.loc[COMP.groupby(["window", "family"]).mae.idxmin()].set_index(["window", "family"])
print(COMP_BEST[["params", "mae"]].round(4).to_string())

W300 comparators done 2199s elapsed
W240 comparators done 2605s elapsed
                                 params     mae
window family                                  
W240   B0b                           {}  3.6181
       B3      {"depth": 3, "lr": 0.05}  0.7359
       B3r     {"depth": 3, "lr": 0.05}  0.6640
       B4                   {"leaf": 5}  0.7102
       B4r                  {"leaf": 1}  0.5678
W300   B0b                           {}  3.6181
       B3      {"depth": 3, "lr": 0.05}  0.9986
       B3r     {"depth": 3, "lr": 0.05}  0.9304
       B4                   {"leaf": 5}  0.9195
       B4r                  {"leaf": 5}  0.8188


In [14]:
# Side-by-side table and decision rules -> compute_plan.json
SIDE = []
for w in PILOT_WINDOWS:
    for kind in ("M1", "M0"):
        for head in ("abs", "res"):
            r = best_row(w, kind, head)
            SIDE.append(dict(window=w, entry=f"{kind}/{head}", config=f"E{int(r.E)} lr{r.lr:g}", mae=float(r.val_mae), gap=float(r.gap)))
    for fam in COMP_GRID:
        if (w, fam) in COMP_BEST.index:
            r = COMP_BEST.loc[(w, fam)]; SIDE.append(dict(window=w, entry=fam, config=r.params, mae=float(r.mae), gap=np.nan))
SIDE = pd.DataFrame(SIDE)
print(SIDE.round(4).to_string(index=False))

DECISION = {}
with guard("decision rules"):
    m_abs = float(np.mean([best_row(w, "M1", "abs").val_mae for w in PILOT_WINDOWS]))
    m_res = float(np.mean([best_row(w, "M1", "res").val_mae for w in PILOT_WINDOWS]))
    head = "res" if m_res <= 0.98 * m_abs else "abs"
    DECISION["head"] = dict(choice=head, mean_best_mae_abs=m_abs, mean_best_mae_res=m_res, res_relative_change=m_res / m_abs - 1.0, rule="res if mean best M1 MAE (res) is at least 2% lower than abs")
    s = PILOT[(PILOT.model == "M1") & (PILOT["head"] == head)].groupby(["E", "lr"]).val_mae.mean().sort_values()
    top = [dict(E=int(E), lr=float(lr), mean_inner_mae=float(v)) for (E, lr), v in s.head(3).items()]
    edge = [dict(E=t["E"], lr=t["lr"], edges=[n for n, bad in (("E", t["E"] in (min(PILOT_E), max(PILOT_E))), ("lr", t["lr"] in (min(PILOT_LR), max(PILOT_LR)))) if bad]) for t in top]
    DECISION["grid_5b"] = dict(pairs=top, edge_info=edge, any_pair_at_grid_edge=any(e["edges"] for e in edge), best_pair_at_grid_edge=bool(edge[0]["edges"]),
                               ranked_by="mean over both windows of M1 inner MAE with the chosen head")
    feas = {}
    for w in PILOT_WINDOWS:
        m1 = float(best_row(w, "M1").val_mae); base = min(float(COMP_BEST.loc[(w, f)].mae) for f in ("B4", "B4r") if (w, f) in COMP_BEST.index)
        feas[w] = dict(m1_best=m1, best_B4_B4r=base, ratio=m1 / base)
    feas["mean_ratio"] = float(np.mean([feas[w]["ratio"] for w in PILOT_WINDOWS]))
    DECISION["pilot_feasibility"] = dict(per_window=feas, note="report only; inner folds, not the week-2 gate (gate target is 0.85 x best baseline on outer-test)")
    DECISION["packing"] = dict(benchmark_choice=PACK, pilot_used=PILOT_PACKING)
    DECISION["seed_sd"] = SEEDS
    print(json.dumps(DECISION, indent=1, default=jd)[:2500])

window  entry                   config    mae    gap
  W300 M1/abs             E120 lr0.004 0.8300 0.1422
  W300 M1/res             E120 lr0.001 0.8347 0.0796
  W300 M0/abs             E120 lr0.004 0.8632 0.2771
  W300 M0/res             E120 lr0.004 0.8435 0.2660
  W300    B0b                       {} 3.6181    NaN
  W300     B3 {"depth": 3, "lr": 0.05} 0.9986    NaN
  W300    B3r {"depth": 3, "lr": 0.05} 0.9304    NaN
  W300     B4              {"leaf": 5} 0.9195    NaN
  W300    B4r              {"leaf": 5} 0.8188    NaN
  W240 M1/abs              E60 lr0.004 0.6849 0.0346
  W240 M1/res             E120 lr0.001 0.6687 0.0636
  W240 M0/abs             E120 lr0.004 0.5680 0.2074
  W240 M0/res             E120 lr0.004 0.5507 0.2050
  W240    B0b                       {} 3.6181    NaN
  W240     B3 {"depth": 3, "lr": 0.05} 0.7359    NaN
  W240    B3r {"depth": 3, "lr": 0.05} 0.6640    NaN
  W240     B4              {"leaf": 5} 0.7102    NaN
  W240    B4r              {"leaf": 1} 0.5678 

## 7. Compute plan for Phase 5b (estimate only)

Time per epoch comes from the benchmark (28 iterations per epoch); epochs for the inner-fold tuning use the pilot iterations per epoch, the final fit scales it by the ratio of outer-train to inner-train cells. M0 is estimated from the G=72 benchmark only (a lower bound at small G). Size-sweep times for c=2 are interpolated geometrically between c=1 and c=4.

In [15]:
def sec_epoch(kind, w, G, c=1.0):
    """Seconds per 28-iteration epoch for G members of `kind` at window w."""
    if kind == "M0":
        r = BENCH[(BENCH.model == "M0")].iloc[0]; return float(r.sec_per_epoch) / 72.0 * G
    b = BENCH[(BENCH.model == "M1") & (BENCH.window == w) & (BENCH.c == 1.0)]
    if PILOT_PACKING == "sequential":
        base = float(b[b.packing == "sequential"].sec_per_epoch.iloc[0]) / 4.0 * G
    else:
        bb = b[(b.packing == "batched")].dropna(subset=["sec_per_epoch"]).sort_values("G")
        base = float(np.interp(G, bb.G.values, bb.sec_per_epoch.values))
    if c != 1.0:
        f4 = float(BENCH[(BENCH.model == "M1") & (BENCH.c == 4.0)].sec_per_epoch.iloc[0] / BENCH[(BENCH.model == "M1") & (BENCH.window == "W300") & (BENCH.c == 1.0) & (BENCH.packing == "batched") & (BENCH.G == 4)].sec_per_epoch.iloc[0])
        base *= {0.5: 1.0, 2.0: math.sqrt(f4), 4.0: f4}[c]
    return base

PLAN = dict(options=[], assumptions={})
with guard("compute plan"):
    ipe_inner = int(run_cfg("W300", "M1", 30, 1e-3)["ipe"])
    mean_inner_cells = float(np.mean([len(f["train_cells"]) for f in FOLDS]))
    ipe_final = int(math.ceil(ipe_inner * len(OUTER_TRAIN) / mean_inner_cells))
    top = DECISION["grid_5b"]["pairs"]; best_E = top[0]["E"]
    full_grid = [(E, lr) for E in PILOT_E for lr in PILOT_LR]
    PLAN["assumptions"] = dict(ipe_inner=ipe_inner, ipe_final=ipe_final, final_seeds=5, tuning_members=4, final_E=best_E, outer_sets=18, packing=PILOT_PACKING,
                               head=DECISION["head"]["choice"], note="windows run in sequence; times are single-GPU minutes")
    def minutes_tune(kind, wins, grid, c=1.0):
        return sum(E * sec_epoch(kind, w, 4, c) * ipe_inner / 28 for w in wins for (E, lr) in grid) / 60
    def minutes_final(kind, wins, c=1.0, E=best_E):
        return sum(E * sec_epoch(kind, w, 5, c) * ipe_final / 28 for w in wins) / 60
    def tree_minutes(wins, fam_grid_sizes):
        s = 0.0
        for fam, ng in fam_grid_sizes.items():
            sp = float(COMP[COMP.family == fam].secs_per_fit.mean())
            s += sp * (NF * ng + 1) * len(wins)
        return s / 60
    for wname, wins in (("priority (W300, W240)", PILOT_WINDOWS), ("all 6 windows", WINDOWS)):
        for kind in ("M1", "M0"):
            for gname, grid in (("3 best pairs", [(t["E"], t["lr"]) for t in top]), ("full 9-pair grid", full_grid)):
                tu, fi = minutes_tune(kind, wins, grid), minutes_final(kind, wins)
                PLAN["options"].append(dict(windows=wname, model=kind, grid=gname, tune_min_per_outer=tu, final_5seeds_min_per_outer=fi, per_outer_total_min=tu + fi, all_18_outer_sets_min=18 * (tu + fi)))
        sw = sum(minutes_final("M1", wins, c=c) for c in (0.5, 1.0, 2.0, 4.0))
        PLAN["options"].append(dict(windows=wname, model="M1 size sweep c in {0.5,1,2,4}", grid="best c=1 config reused", tune_min_per_outer=0.0, final_5seeds_min_per_outer=sw, per_outer_total_min=sw, all_18_outer_sets_min=18 * sw))
        tm = tree_minutes(wins, {"B3r": len(COMP_GRID["B3r"]), "B4r": len(COMP_GRID["B4r"])})
        PLAN["options"].append(dict(windows=wname, model="B3r + B4r (CPU, same grids as pilot)", grid="pilot grids", tune_min_per_outer=tm, final_5seeds_min_per_outer=0.0, per_outer_total_min=tm, all_18_outer_sets_min=18 * tm))
    OPT = pd.DataFrame(PLAN["options"]); print(OPT.round(1).to_string(index=False))
cp = dict(**{k: DECISION.get(k) for k in ("head", "grid_5b", "pilot_feasibility", "packing", "seed_sd")}, compute_plan=PLAN)
(OUT / "compute_plan.json").write_text(json.dumps(cp, indent=2, default=jd))

              windows                                model                   grid  tune_min_per_outer  final_5seeds_min_per_outer  per_outer_total_min  all_18_outer_sets_min
priority (W300, W240)                                   M1           3 best pairs                 2.5                         1.0                  3.5                   62.4
priority (W300, W240)                                   M1       full 9-pair grid                 6.6                         1.0                  7.5                  135.4
priority (W300, W240)                                   M0           3 best pairs                 0.1                         0.0                  0.1                    1.4
priority (W300, W240)                                   M0       full 9-pair grid                 0.1                         0.0                  0.2                    3.1
priority (W300, W240)       M1 size sweep c in {0.5,1,2,4} best c=1 config reused                 0.0                         6.3 

6685

## 8. Figures

In [16]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
FIGS = []
with guard("figure benchmark"):
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for w in WINDOWS:
        b = BENCH[(BENCH.window == w) & (BENCH.model == "M1") & (BENCH.c == 1.0) & (BENCH.packing == "batched")].sort_values("G")
        ln, = ax.plot(b.G, b.member_epochs_per_s, marker="o", label=w)
        s = BENCH[(BENCH.window == w) & (BENCH.packing == "sequential")].member_epochs_per_s
        if len(s): ax.axhline(float(s.iloc[0]), color=ln.get_color(), ls=":", lw=0.8)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("members per batched run (G)"); ax.set_ylabel("members x epochs / s")
    ax.set_title("M1 (c=1) throughput; dotted = sequential single models"); ax.legend(ncol=2, fontsize=8); fig.tight_layout()
    fig.savefig(FIG / "benchmark_throughput.png", dpi=130); plt.close(fig); FIGS.append("benchmark_throughput.png")

with guard("figure learning curves"):
    fig, axes = plt.subplots(1, len(PILOT_WINDOWS), figsize=(11, 4), sharey=False)
    for ax, w in zip(np.atleast_1d(axes), PILOT_WINDOWS):
        for kind, colr in (("M1", "C0"), ("M0", "C1")):
            b = best_row(w, kind); o = run_cfg(w, kind, int(b.E), float(b.lr)); hi = o["heads"].index(b["head"])
            ep = [c_["epoch"] for c_ in o["ckpts"]]
            ax.plot(ep, [c_["val"][hi].mean() for c_ in o["ckpts"]], color=colr, marker="o", label=f"{kind} val ({b['head']}, E{int(b.E)}, lr {b.lr:g})")
            ax.plot(ep, [c_["train"][hi].mean() for c_ in o["ckpts"]], color=colr, ls="--", label=f"{kind} train")
        ax.set_title(f"{w}: inner cell-macro MAE (pp)"); ax.set_xlabel("epoch"); ax.set_ylabel("MAE (pp)"); ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(FIG / "pilot_learning_curves.png", dpi=130); plt.close(fig); FIGS.append("pilot_learning_curves.png")

with guard("figure heatmaps"):
    hd = DECISION["head"]["choice"]
    fig, axes = plt.subplots(1, len(PILOT_WINDOWS), figsize=(9, 3.6))
    for ax, w in zip(np.atleast_1d(axes), PILOT_WINDOWS):
        pv = PILOT[(PILOT.window == w) & (PILOT.model == "M1") & (PILOT["head"] == hd)].pivot(index="E", columns="lr", values="val_mae")
        im = ax.imshow(pv.values, cmap="viridis_r", aspect="auto")
        ax.set_xticks(range(pv.shape[1])); ax.set_xticklabels([f"{v:g}" for v in pv.columns]); ax.set_yticks(range(pv.shape[0])); ax.set_yticklabels(pv.index)
        for i in range(pv.shape[0]):
            for j in range(pv.shape[1]): ax.text(j, i, f"{pv.values[i, j]:.3f}", ha="center", va="center", color="w", fontsize=8)
        ax.set_title(f"M1 {w} ({hd} head)"); ax.set_xlabel("lr"); ax.set_ylabel("epochs"); fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout(); fig.savefig(FIG / "pilot_heatmap_M1.png", dpi=130); plt.close(fig); FIGS.append("pilot_heatmap_M1.png")
print(FIGS)

['benchmark_throughput.png', 'pilot_learning_curves.png', 'pilot_heatmap_M1.png']


## 9. Report

In [17]:
# end-of-run integrity checks, report files, printed summary
check("week2_gate.json sha256 == gate_hash.txt (end)", gate_ok(), "unchanged")
check("soh_common.py unchanged by this notebook", sha256_file(Path(sc.__file__)) == SC_FILE_HASH, "read-only")
RUNTIME_MIN = (time.time() - T_START) / 60
check("runtime within about 40 minutes", RUNTIME_MIN <= 45, f"{RUNTIME_MIN:.1f} min (cached steps are fast on rerun)", level="WARN")

def table(df, **kw):
    return df.to_string(index=False, float_format=lambda v: f"{v:.4f}", **kw)
L = []; add = L.append
add("PHASE 5A REPORT"); add("=" * 100)
add(f"runtime: {RUNTIME_MIN:.1f} min | repeat {R0}, outer fold {O0} | inner folds only | seed {SEED}")
add("GPU: " + ", ".join(f"{k}={v}" for k, v in GPU_FACTS.items()))
add(f"parameters: M1 c=0.5/1/2/4 = {cnt[0.5]}/{cnt[1]}/{cnt[2]}/{cnt[4]} | M0 = {n_m0}")
add(""); add("TESTS")
for c_ in CHECKS:
    if c_["name"].startswith("4"): add(f"  [{c_['status']}] {c_['name']} :: {c_['detail'][:110]}")
add(""); add("BENCHMARK (members x epochs/s, ms per iteration, peak GB)")
bshow = BENCH.assign(thr=BENCH.member_epochs_per_s)[["window", "model", "c", "packing", "G", "ms_per_iter", "member_epochs_per_s", "peak_mem_gb"]]
add(table(bshow)); add(f"packing choice: {PACK}; used in pilot: {PILOT_PACKING}")
add(""); add("PILOT: best config per model/head/window versus comparators on the same inner folds (cell-macro MAE, pp)")
add(table(SIDE))
add(""); add("SEED SD (M1 best config, 4 seeds, mean inner MAE)")
for w, s_ in SEEDS.items(): add(f"  {w}: head {s_['head']} E{s_['E']} lr{s_['lr']:g} mean {s_['mean']:.4f} sd {s_['sd']:.4f}")
add(""); add("SIZE SWEEP W300 (M1)")
for c_, s_ in SIZE.items(): add(f"  c={c_}: params {s_['n_params']}, inner MAE {s_['mae']:.4f}")
add(""); add("DECISIONS (compute_plan.json)")
add(f"  head: {DECISION['head']['choice']} (res vs abs change {100 * DECISION['head']['res_relative_change']:+.1f}%)")
add("  5b grid: " + "; ".join(f"E{p['E']} lr{p['lr']:g} ({p['mean_inner_mae']:.4f})" for p in DECISION["grid_5b"]["pairs"]) + f" | any pair at edge: {DECISION['grid_5b']['any_pair_at_grid_edge']}, best pair at edge: {DECISION['grid_5b']['best_pair_at_grid_edge']}")
add("  feasibility (M1 / best of B4,B4r): " + ", ".join(f"{w} {DECISION['pilot_feasibility']['per_window'][w]['ratio']:.3f}" for w in PILOT_WINDOWS) + f" | mean {DECISION['pilot_feasibility']['per_window']['mean_ratio']:.3f}")
add(""); add("COMPUTE PLAN (minutes)"); add(table(OPT) if "OPT" in globals() else "  (not available)")
for st in ("PASS", "WARN", "FAIL"):
    items = [c_ for c_ in CHECKS if c_["status"] == st]
    add(""); add(f"CHECKS {st} ({len(items)})")
    for c_ in items: add(f"  {c_['name']} :: {c_['detail'][:100]}")
add(""); add(f"ERRORS ({len(ERRORS)})")
for e in ERRORS: add(f"  {e['where']}: {e['error'][:160]}")
arts = sorted(p for p in OUT.rglob("*") if p.is_file() and "cache" not in p.parts) + [SD]
add(""); add("ARTIFACTS")
for p in arts: add(f"  {p.relative_to(ROOT)}  {p.stat().st_size / 1e3:.1f} KB")
add(f"  (cache: {sum(1 for _ in CACHE.glob('*'))} files)")
REPORT = "\n".join(L)
(OUT / "phase5a_report.txt").write_text(REPORT, encoding="utf-8")
VER = dict(runtime_min=RUNTIME_MIN, gpu=GPU_FACTS, param_counts=dict(M1={str(k): v for k, v in cnt.items()}, M0=n_m0), checks=CHECKS, errors=ERRORS,
           hashes=dict(gate_json=sha256_file(GATE_FILE), gate_hash_file=GATE_HASH_FILE, soh_common=SC_FILE_HASH, kept_cells=KEPT_HASH, soh_deep=sha256_file(SD)),
           benchmark=BENCH.to_dict("records"), pilot_best=SIDE.to_dict("records"), seeds=SEEDS, size=SIZE, decisions=DECISION,
           split=dict(repeat=R0, outer=O0, n_outer_train_cells=len(OUTER_TRAIN), n_outer_test_cells_never_loaded=len(OUTER_TEST), n_dev_rows=len(DEV)),
           artifacts=[str(p.relative_to(ROOT)) for p in arts])
(OUT / "verification_report.json").write_text(json.dumps(VER, indent=1, default=jd))
print(REPORT)

[PASS] week2_gate.json sha256 == gate_hash.txt (end): unchanged
[PASS] soh_common.py unchanged by this notebook: read-only
[PASS] runtime within about 40 minutes: 43.6 min (cached steps are fast on rerun)
PHASE 5A REPORT
runtime: 43.6 min | repeat 0, outer fold 0 | inner folds only | seed 42
GPU: torch=2.14.1+cu130, cuda=13.0, cudnn=92400, device=NVIDIA GeForce RTX 3050 6GB Laptop GPU, capability=8.6, vram_total_gb=6.44, vram_free_gb=5.41, tf32_matmul=False, tf32_cudnn=False
parameters: M1 c=0.5/1/2/4 = 817/2977/11329/44161 | M0 = 2969

TESTS
  [PASS] 4 inner folds :: 4
  [PASS] 4a M1(c=1) has exactly 2,977 parameters :: 2977
  [PASS] 4a size ratios about 0.3 / 1 / 3.7 / 15 (D36) :: {0.5: 0.27, 1: 1.0, 2: 3.81, 4: 14.83}
  [PASS] 4a M0 parameter count close to M1 :: 2969
  [PASS] 4b M1: batched == single models in eval mode (init) :: max abs diff 1.49e-08
  [FAIL] 4b M1: weights equal after 20 identical steps :: max abs weight diff 1.43e-03; eval output diff 1.13e-04
  [PASS] 4b M0: ba